#### Extraction

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import math
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
 
from Extraction import extract
from Transformation_Pretraitement import preprocessing_polars
from inceptionTimeModified import (
    evaluate_on_test,
    load_model_from_checkpoint,
    predict_proba,
    train_inception_time,
)
import utils_inception as ui

In [ ]:
pl.Config.set_tbl_cols(-1)

 ##### Dataframe statique

In [ ]:
path = "../Datasets/clean_full_static_ano.parquet"
df_static = pl.read_parquet(path)
df_static = df_static.with_columns(pl.col("encounterId").cast(pl.Int32))

In [ ]:
# df_static = df_static.with_columns(
#     pl.when(pl.col("deces_datediff_days").is_between(-1, 0))
#     .then(0)
#     .otherwise(pl.col("deces_datediff_days"))
#     .alias("deces_datediff_days")
# ).filter(
#     (pl.col("deces_datediff_days") >= 0) |
#     (pl.col("deces_datediff_days").is_null())
# )


In [ ]:
df_static = df_static.with_columns(
    pl.when(pl.col("deces_datediff_days").is_between(-1, 0))
    .then(0)
    .otherwise(pl.col("deces_datediff_days"))
    .alias("deces_datediff_days")
)


In [ ]:
df_static["isDeceased"].describe()

##### Dataframe dynamique

In [ ]:
path = "../Datasets/df_with_calculated_features.parquet"
df_test = extract.extract_data_survie(path)

#### Transformation_Prétraitement

Il faudra changer hour_offset pour pouvoir prendre une date fixe et non juste un temps en arrière

ajout de la colonne age qui est dans le thesaurus

In [ ]:
df_test = df_test.join(
    df_static[["encounterId", "age"]],
    on="encounterId",
    how="left"
)

On utilise ici le preprocessing de Gabrielle mais avec l'optimisation polars réalisée par mes soins, puisque l'ancien code mettait beaucoup trop de temps à tourner.

In [ ]:
df_clean = preprocessing_polars.prepare_data(df_test,0, strict_mode = True)
df_clean

In [ ]:
df_with_idx = df_clean.with_row_index("idx")
 
idx = (

    df_with_idx

    .filter(pl.col("fio2_corr").is_null())

    .select("idx")

)
 
print(len(idx))

df_clean.filter(pl.col("fio2_corr").is_null())

Il faut rajouter isDeceased sinon on n'a pas de Y

In [ ]:
df_clean = df_clean.join(
    df_static[["encounterId", "isDeceased", "deces_datediff_days"]],
    on="encounterId",
    how="inner"
)

On rajoute l'étiquette z qui vaut 0 si deces_datediff_days == null, 1 si < 24 après la fin de la fenêtre et 2 si > 24

In [ ]:
df_clean = df_clean.with_columns((pl
    .when(pl.col("deces_datediff_days")
        .is_null())
        .then(pl.lit(0))
    .when(((pl.col("deces_datediff_days") * 24) + (pl.col("heure_calibree").min().over("encounterId"))).is_between(0, 24))
        .then(pl.lit(1))
    .when(((pl.col("deces_datediff_days") * 24) + (pl.col("heure_calibree").min().over("encounterId"))).is_between(24, 672))
        .then(pl.lit(2))
    .when(((pl.col("deces_datediff_days") * 24) + (pl.col("heure_calibree").min().over("encounterId"))).is_between(672, 2190))
        .then(pl.lit(3))
    .otherwise(pl.lit(4))).alias("DeceasedTimeType"))
df_clean = df_clean.drop("deces_datediff_days")

In [ ]:
df_clean["DeceasedTimeType"].describe()

à titre indicatif : 

In [ ]:
print("nombre d'enregistrement de patients vivants (isDeceased)", len(df_clean.filter(pl.col("isDeceased") == 0)))
print("nombre d'enregistrement de patients vivants (DeceasedTimeType)", len(df_clean.filter(pl.col("DeceasedTimeType") == 0)))
print("nombre d'enregistrement de patients morts (isDeceased)", len(df_clean.filter(pl.col("isDeceased") == 1)))
print("nombre d'enregistrement de patients morts (DeceasedTimeType)", len(df_clean.filter(pl.col("DeceasedTimeType")> 0)))
print("nombre d'enregistrement de patients morts moins de 24 heures après la fin de la fenêtre", len(df_clean.filter(pl.col("DeceasedTimeType") == 1)))
print("nombre d'enregistrement de patients morts moins de 28 jours après la fin de la fenêtre", len(df_clean.filter(pl.col("DeceasedTimeType") == 2)))
print("nombre d'enregistrement de patients morts moins de 3 mois après la fin de la fenêtre", len(df_clean.filter(pl.col("DeceasedTimeType") == 3)))
print("nombre d'enregistrement de patients morts plus de 3 mois après la fin de la fenêtre", len(df_clean.filter(pl.col("DeceasedTimeType") == 4)))

En pratique, on utilisera une simplification : 

In [ ]:
df_clean = df_clean.with_columns((pl
    .when(pl.col("DeceasedTimeType") > 2)
        .then(pl.lit(0))
    .otherwise((pl.col("DeceasedTimeType")))).alias("DeceasedTimeType"))

In [ ]:
df_clean["DeceasedTimeType"].describe()

In [ ]:
print("nombre d'enregistrement de patients vivants (DeceasedTimeType)", len(df_clean.filter(pl.col("DeceasedTimeType") == 0)))
print("nombre d'enregistrement de patients morts moins de 24 heures après la fin de la fenêtre", len(df_clean.filter(pl.col("DeceasedTimeType") == 1)))
print("nombre d'enregistrement de patients morts moins de 28 jours après la fin de la fenêtre", len(df_clean.filter(pl.col("DeceasedTimeType") == 2)))

#### Préparation pour InceptionTime

##### Split train/test + scaling + reshape

In [ ]:
patient_col="encounterId"
time_col="heure_calibree"
target_col="isDeceased"
expected_length=24

# On garde les encounters de longueur exacte 
valid_ids = (df_clean.group_by(patient_col).len()
        .filter(pl.col("len") == expected_length)
        .select(patient_col))

df_clean = df_clean.join(valid_ids, on=patient_col, how="inner")

if df_clean.is_empty():
    raise ValueError("Aucun patient n'a exactement la longueur attendue.")

# Tri obligatoire (même si en théorie il est déjà fait)
df_clean = df_clean.sort(patient_col, time_col)

# On prépare le jeu d'entraînement
X = df_clean.select(pl.exclude(target_col, patient_col, "DeceasedTimeType")).to_numpy()
y = df_clean[target_col].to_numpy()
groups = df_clean[patient_col].to_numpy() # grouper en fonction d'un individu

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# On prend un split (comme train/test mais adapté aux individus)
train_idx, test_idx = next(sgkf.split(X=X, y=y, groups=groups))

# Normalement pas besoin de sort mais soyons prudents...
train_df = df_clean[train_idx].sort([patient_col, time_col])
test_df = df_clean[test_idx].sort([patient_col, time_col])
# Ok maintenant, on applique le scaler sur le dataframe

train_pd, test_pd = ui.scaling(train_df, test_df)

# On retransforme en df polars
train_df, test_df = pl.from_pandas(train_pd), pl.from_pandas(test_pd)

# On construit la séquence attendue (N, T, F) à partir des deux dataframes train/test
X_train_3d, y_train_seq = ui.build_sequences(
    train_df, patient_col, target_col, expected_length
)

X_test_3d, y_test_seq = ui.build_sequences(
    test_df, patient_col, target_col, expected_length
)

In [ ]:
train_df.group_by("encounterId").len().describe()

In [ ]:
train_df.describe()

#### Recherche des meilleurs hyperparamètres avec Optuna

##### Test pour faire fonctionner le dashboard

In [ ]:

study = optuna.create_study(
    study_name="test_optuna",
    direction="minimize",
    storage="sqlite:///optuna_inceptiontest.db",
    load_if_exists=True,
)
 
study.optimize(lambda trial: trial.suggest_float("x", 0, 1), n_trials=1)

##### Stage 1 : recherche large, en utilisant les fonctions : extract_best_val_loss, make_objective_stage1, run_stage1_search (fonction principale)

In [ ]:
fixed_params = {
    "val_ratio": 0.2,
    "epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "calibrate": False,
    "device": "cuda",
}
 
study_stage1 = ui.run_stage1_search(
    X_train_3d,
    y_train_seq,
    n_trials=40,
    study_name="inception_stage1",
    storage="sqlite:///optuna_inceptiontest.db",
    metric_name="val_loss",   # ou "val_f1", selon ton history
    fixed_params=fixed_params,
)

#### Training sur InceptionTime

In [ ]:
model, T, history, splits = train_inception_time(
    X_train_3d, y_train_seq,
    num_blocks = 6,
    out_channels = 32,
    bottleneck_channels = 8,
    kernel_sizes = 21,
    batch_size = 16,
    lr = 0.0009572131781501278,
    weight_decay = 1.216426840149487e-06,
    clip_grad = 0.5,
    use_scheduler = False,
    epochs=100,
    patience=10,
    save_best_path="models/inception_test2_final2_clean.pt"
)

#### Evaluation des résultats obtenus

In [ ]:
auc, brier, T = evaluate_on_test(
    X_test_3d, y_test_seq,
    "models/inception_test2_final2_clean.pt"
)

model, _, T = load_model_from_checkpoint("models/inception_test2_final2_clean.pt")

In [ ]:
probas = predict_proba(model, X_test_3d, T=T)
print(probas)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_seq, probas)
auc = roc_auc_score(y_test_seq, probas)
 
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Hasard")
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs")
plt.title("Courbe ROC")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
 
sns.kdeplot(probas[y_test_seq == 0], label="Survivants", fill=True)
sns.kdeplot(probas[y_test_seq == 1], label="Décès", fill=True)
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Densité")
plt.title("Distribution des scores (KDE)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
y_test = y_test_seq

plt.figure()
 
# Survivants
data_0 = probas[y_test == 0]
sns.kdeplot(data_0, color="lightblue")
x0, y0 = plt.gca().lines[-1].get_data()
y0 = y0 * len(data_0)  # conversion densité → counts
plt.plot(x0, y0, color="blue", label="Survivants")
plt.fill_between(x0, y0, alpha=0.3, color="lightblue")
 
# Décès
data_1 = probas[y_test == 1]
sns.kdeplot(data_1, color="orange")
x1, y1 = plt.gca().lines[-1].get_data()
y1 = y1 * len(data_1)
plt.plot(x1, y1, color="orange", label="Décès")
plt.fill_between(x1, y1, alpha=0.3, color="orange")
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Nombre de patients")
plt.title("Distribution des scores")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.style.use("seaborn-v0_8")
plt.show()

In [ ]:

prob_true, prob_pred = calibration_curve(y_test, probas, n_bins=10)
 
plt.figure()
plt.plot(prob_pred, prob_true, marker="o", label="Modèle")
plt.plot([0, 1], [0, 1], "--", label="Calibration idéale")
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Fréquence observée")
plt.title("Calibration curve")
plt.legend()
plt.grid()
plt.show()

In [ ]:
thresholds = np.linspace(0.1, 0.9, 50)
f1s = []
best_f1 = 0
for t in thresholds:
    y_pred = (probas >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred))
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t
 
import matplotlib.pyplot as plt
 
plt.plot(thresholds, f1s)
plt.xlabel("Threshold")
plt.ylabel("F1 score")
plt.title("F1 vs Threshold")
plt.grid()
plt.show()

print(f"Le meilleur f1 score de{best_f1 : .2f} est atteint lorsque le threshold est égal à{best_t : .2f}")

In [ ]:
threshold = 0.44
y_pred = (probas >= threshold).astype(int)
 
cm = confusion_matrix(y_test, y_pred)
 
plt.figure()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title(f"Confusion matrix (threshold={threshold})")
plt.show()